---
title: Modern Deep Learning Methods
---

## Modeling Approach

We will use the pretrained EfficientNet-B3 and Resnet-50 from the PyTorch library. These models were pre-trained on ImageNet and provide strong feature extraction capabilities that transfer well to medical imaging tasks. We replace the final classification layers to output 5 classes corresponding to diabetic retinopathy severity levels (0-4: No DR, Mild, Moderate, Severe, Proliferative DR).

In [1]:
import pandas as pd
import numpy as np
import os
from PIL import Image
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
import torchvision.models as models
from torch.utils.data import DataLoader, Subset, TensorDataset
import torch.nn.functional as F
import cv2
from sklearn.model_selection import train_test_split
from sklearn.metrics import cohen_kappa_score
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')

In [2]:
x_train = torch.load('/kaggle/input/aptos-preprocess-train/x_train.pt')
y_train = torch.load('/kaggle/input/aptos-preprocess-train/y_train.pt')
x_test = torch.load('/kaggle/input/aptos-preprocess-train/x_test.pt')
y_test = torch.load('/kaggle/input/aptos-preprocess-train/y_test.pt')

In [3]:
def qwk(y_true, y_pred, num_classes=5):
    return cohen_kappa_score(y_true, y_pred, weights='quadratic')

### EfficientNet

This implementation adapts EfficientNet-B3 by replacing the original single-layer classifier with a custom two-layer fully connected head. The modified classifier includes an initial dropout layer (30%) for regularization, followed by a 512-neuron hidden layer with ReLU activation, another dropout layer, and finally a 5-neuron output layer corresponding to the five DR severity classes. This deeper classifier head allows the model to better map the rich feature representations from EfficientNet's backbone to the nuanced ordinal classification task of DR severity assessment, while the dual dropout layers help prevent overfitting on medical imaging datasets.

In [4]:
def create_efficientnet(num_classes=5, dropout_rate=0.3):
    model = models.efficientnet_b3(pretrained=True)
    
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(dropout_rate),
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Dropout(dropout_rate),
        nn.Linear(512, num_classes)
    )
    return model

### ResNet

Similarly, this ResNet-50 implementation replaces the original single fully connected layer with a custom two-layer classifier head optimized for diabetic retinopathy classification. The modified architecture includes an initial dropout layer (50%) for strong regularization, a 512-neuron hidden layer with ReLU activation, a second dropout layer with reduced rate (25%), and a final 5-neuron output layer for DR severity classification. The higher initial dropout rate (50% vs 30% in EfficientNet) and progressive dropout reduction (50% → 25%) provides more aggressive regularization to leverage ResNet-50's deeper architecture while preventing overfitting.

In [5]:
def create_resnet(num_classes=5, dropout_rate=0.5):
    model = models.resnet50(pretrained=True)
    
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(dropout_rate),
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Dropout(dropout_rate/2),
        nn.Linear(512, num_classes)
    )
    return model

We will use the same training and evaluation code that we used for our baseline CNN.

In [6]:
def train_model(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(train_loader)

In [7]:
def evaluate_model(model, test_loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
    
    return np.array(all_labels), np.array(all_preds)

In [8]:
def main(model, x_train, y_train, x_test, y_test):    
    print(f"Train set: {len(x_train)} images")
    print(f"Test set: {len(x_test)} images")
    
    train_dataset = TensorDataset(x_train, y_train)
    test_dataset = TensorDataset(x_test, y_test)
    
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    
    print(f"Using device: {device}")
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    # Training loop
    epochs = 20
    for epoch in range(epochs):
        train_loss = train_model(model, train_loader, criterion, optimizer, device)
        
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Loss: {train_loss:.4f}")
    
    y_true, y_pred = evaluate_model(model, test_loader, device)
    
    kappa = qwk(y_true, y_pred)
    accuracy = (y_true == y_pred).mean()
    
    print(f"\nFinal Results:")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Quadratic Weighted Kappa: {kappa:.4f}")
    
    # Class distribution
    print(f"\nTrue labels distribution: {np.bincount(y_true)}")
    print(f"Predicted labels distribution: {np.bincount(y_pred)}")
    
    return model, kappa

In [9]:
efficientnet = create_efficientnet()
efficientnet_model, kappa = main(efficientnet, x_train, y_train, x_test, y_test)

Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth
100%|██████████| 47.2M/47.2M [00:00<00:00, 174MB/s] 


Train set: 2929 images
Test set: 733 images
Using device: cuda
Model parameters: 11,485,741
Epoch 5/20, Loss: 0.3105
Epoch 10/20, Loss: 0.1606
Epoch 15/20, Loss: 0.1247
Epoch 20/20, Loss: 0.0844

Final Results:
Accuracy: 0.7817
Quadratic Weighted Kappa: 0.8538

True labels distribution: [361  74 200  39  59]
Predicted labels distribution: [368  72 194  35  64]


In [10]:
torch.save(efficientnet_model, 'efficientnet.pth')

In [11]:
resnet = create_resnet()
resnet_model, kappa = main(resnet, x_train, y_train, x_test, y_test)

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 196MB/s]


Train set: 2929 images
Test set: 733 images
Using device: cuda
Model parameters: 24,559,685
Epoch 5/20, Loss: 0.5646
Epoch 10/20, Loss: 0.4825
Epoch 15/20, Loss: 0.3752
Epoch 20/20, Loss: 0.2641

Final Results:
Accuracy: 0.7149
Quadratic Weighted Kappa: 0.6986

True labels distribution: [361  74 200  39  59]
Predicted labels distribution: [359 185 184   4   1]


In [12]:
torch.save(resnet_model, 'resnet.pth')